# Diamond Price Valuation System — 10-Fold Cross Validation (Full Dataset)

> 🌐 **Live Cloud Deployment URL**: [https://diamond-price-valuation.streamlit.app/](https://diamond-price-valuation.streamlit.app/)
> *(Integrated Streamlit Web Application deployed on the cloud)*

**Course**: Machine Learning Fundamentals (AIC354)  
**Institution**: COMSATS University Islamabad, Lahore Campus  
**Instructor**: Dr. Rao Muhammad Adeel Nawab  
**Task**: Assignment 3 — Applying, Evaluating, and Comparing Multiple ML Algorithms on Large Real-World Dataset  


## Setup: Running This Notebook on Google Colab

This notebook is ready to run end-to-end on Google Colab:
1. Upload this notebook to [Google Colab](https://colab.research.google.com).
2. Click **Runtime** > **Run all** (or press `Ctrl+F9`).
3. When prompted in Step 2.1, upload `diamonds.csv`.
4. The entire 10-fold cross validation, tables, graphs, final model training, and sample inference will execute automatically.

## Introduction

##### **Definition**
A regression machine learning system that predicts the market price (in USD) of a diamond from its physical attributes and the internationally recognized **4Cs**: Carat weight, Cut quality, Color grade, Clarity grade, and precise 3-dimensional measurements ($X, Y, Z$, Depth %, Table %).

##### **Application**
Retail jewelry valuation, luxury goods appraisal, insurance underwriting, pawnbroking, e-commerce marketplace pricing, and commodity investment.

### Aim of This Notebook
Evaluate and compare **5 distinct machine learning algorithms** (Ridge Regression, Linear SVR, AdaBoost, Extra Trees, and Histogram Gradient Boosting) using **K-Fold Cross-Validation** to establish the most accurate and reliable production model.

### Evaluation Measure: Root Mean Squared Error (RMSE)
$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$
RMSE is measured in US Dollars ($), penalizing large estimation errors heavily, which is crucial for high-value gemstones.

# Code — Diamond Price Valuation System

In [ ]:
# Libraries
import os
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Preprocessing and pipeline
from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error

# 5 candidate algorithms
from sklearn.linear_model import Ridge
from sklearn.svm import LinearSVR
from sklearn.ensemble import AdaBoostRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor

# Environment check
IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local / Jupyter'}")

# Step 2: Load Sample Data
## Step 2.1: Upload the Dataset (Google Colab)

In [ ]:
DATA_FILE = 'diamonds.csv'

if IN_COLAB and not os.path.exists(DATA_FILE):
    from google.colab import files
    print('Please upload diamonds.csv:')
    uploaded = files.upload()
    DATA_FILE = next(iter(uploaded))
elif not os.path.exists(DATA_FILE):
    print('Local diamonds.csv not found; fetching via seaborn...')
    df_raw = sns.load_dataset('diamonds')
    df_raw.to_csv(DATA_FILE, index=False)

print(f"Dataset file ready: '{DATA_FILE}'")

## Step 2.2: Read the Dataset

In [ ]:
df = pd.read_csv(DATA_FILE)
# Remove zero dimension records
df = df[(df['x'] > 0) & (df['y'] > 0) & (df['z'] > 0)].copy()
print(f"Loaded dataset shape: {df.shape[0]:,} rows and {df.shape[1]} columns")
df.head()

# Step 3: Understand and Pre-process Sample Data
## Step 3.1: Understand Sample Data

In [ ]:
print('Dataset Info:')
df.info()
print('\nMissing Values:')
print(df.isnull().sum())
print('\nDescriptive Statistics:')
display(df.describe().T)

### Understanding the Output (price in USD)
Visualizing the target variable distribution and checking for skewness.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['price'], kde=True, ax=axes[0], color='#2563eb', bins=40)
axes[0].set_title('Diamond Price Distribution (USD)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Price ($)')

sns.boxplot(x=df['price'], ax=axes[1], color='#38bdf8')
axes[1].set_title('Diamond Price Boxplot (Outlier Analysis)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Price ($)')
plt.tight_layout()
plt.show()

print(f"Price Summary: Min=${df['price'].min():,.2f}, Median=${df['price'].median():,.2f}, Mean=${df['price'].mean():,.2f}, Max=${df['price'].max():,.2f}")

> **Observation**: Diamond price is positively skewed with a long tail of ultra-luxury gems. Standardizing features and evaluating robust ensembles like Extra Trees and Histogram Gradient Boosting handles these non-linearities effectively.

# Step 4: Feature Extraction
Separating features into continuous numeric attributes and nominal categorical attributes.

In [ ]:
numeric_features = ['carat', 'depth', 'table', 'x', 'y', 'z']
categorical_features = ['cut', 'color', 'clarity']

X = df[numeric_features + categorical_features].copy()
y = df['price'].copy()

print(f"Predictor Features ({len(numeric_features) + len(categorical_features)}):")
print(f"  Numeric ({len(numeric_features)})    : {numeric_features}")
print(f"  Categorical ({len(categorical_features)}): {categorical_features}")
print(f"Target Vector (y): price (${y.min():,.0f} to ${y.max():,.0f})")

# Step 5: Build the Pre-processing Pipeline

### Why put pre-processing inside the pipeline?
To prevent **data leakage**, transformers must only learn statistics (means, standard deviations, categorical levels) from the training folds and apply them to validation folds without peeking ahead.

In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ]
)

print('Pre-processing pipeline defined successfully.')

# Step 6: Define the 5 Regression Algorithms

**Note**: Per assignment instructions, these 5 algorithms are **completely distinct** from the 5 algorithms used in the Task 1 Car Price benchmark (Linear Regression, KNN, Decision Tree, Random Forest, Gradient Boosting):
1. **Ridge Regression**: L2-regularized linear model that handles multicollinearity among physical dimensions.
2. **Linear SVR**: Epsilon-support vector regression with linear kernel.
3. **AdaBoost Regressor**: Adaptive boosting ensemble with sequential tree corrections.
4. **Extra Trees Regressor**: Extremely randomized tree ensemble for lower variance.
5. **Histogram Gradient Boosting**: High-speed, histogram-binned gradient boosting regressor.

In [ ]:
algorithms = {
    'Ridge Regression': Ridge(alpha=1.0),
    'Linear SVR': LinearSVR(random_state=42, max_iter=2000),
    'AdaBoost': AdaBoostRegressor(n_estimators=50, random_state=42),
    'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1, max_depth=20),
    'Hist Gradient Boosting': HistGradientBoostingRegressor(random_state=42)
}

print('5 Candidate Algorithms Configured:')
for idx, name in enumerate(algorithms, start=1):
    print(f"  {idx}. {name}")

# Step 7: Set Up 10-Fold Cross Validation
Splitting the data into 10 balanced, shuffled folds with `random_state=42` for reproducibility.

In [ ]:
K = 10
kf = KFold(n_splits=K, shuffle=True, random_state=42)
print(f"K-Fold Cross-Validation configured with K={K} folds.")

# Step 8: Run 10-Fold Cross Validation for All 5 Algorithms
Evaluating validation RMSE on each fold with a `tqdm` progress bar.

In [ ]:
fold_rmse = {name: [] for name in algorithms}
training_times = {name: 0.0 for name in algorithms}

total_iterations = len(algorithms) * K
progress_bar = tqdm(total=total_iterations, desc='Running Cross Validation')

for algo_name, algorithm in algorithms.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', algorithm)
    ])
    start_time = time.time()
    for fold_num, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        pipeline.fit(X_train, y_train)
        preds = pipeline.predict(X_val)
        rmse = float(root_mean_squared_error(y_val, preds))
        fold_rmse[algo_name].append(rmse)
        progress_bar.update(1)
    training_times[algo_name] = time.time() - start_time

progress_bar.close()
print('Cross-validation complete across all algorithms.')

# Step 9: Compare the Algorithms
## Step 9.1: RMSE of Each Algorithm on Each Fold

In [ ]:
per_fold_table = pd.DataFrame(fold_rmse, index=[f'Fold {i}' for i in range(1, K + 1)])
per_fold_table.style.format('${:,.2f}')

## Step 9.2: Summary Comparison Table

In [ ]:
avg_price = y.mean()
summary = pd.DataFrame({
    'Mean RMSE ($)': per_fold_table.mean(),
    'Std RMSE ($)': per_fold_table.std(),
    'Error % of Mean Price': (per_fold_table.mean() / avg_price) * 100,
    'Total Runtime (s)': pd.Series(training_times)
}).sort_values('Mean RMSE ($)')

summary.style.format({
    'Mean RMSE ($)': '${:,.2f}',
    'Std RMSE ($)': '${:,.2f}',
    'Error % of Mean Price': '{:.2f}%',
    'Total Runtime (s)': '{:.1f}s'
})

## Step 9.3: Visual Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Boxplot of fold RMSE distributions
sns.boxplot(data=per_fold_table, ax=axes[0], palette='Blues_r')
axes[0].set_title(f'RMSE Spread Across 10 Folds', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Validation RMSE ($)')
axes[0].tick_params(axis='x', rotation=25)

# Bar chart with standard deviation error bars
sorted_summary = summary.sort_values('Mean RMSE ($)')
axes[1].bar(sorted_summary.index, sorted_summary['Mean RMSE ($)'], yerr=sorted_summary['Std RMSE ($)'],
           capsize=5, color='#2563eb', alpha=0.85, edgecolor='black')
axes[1].set_title('Mean RMSE by Algorithm (Lower is Better)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Mean RMSE ($)')
axes[1].tick_params(axis='x', rotation=25)
for i, v in enumerate(sorted_summary['Mean RMSE ($)']):
    axes[1].text(i, v + 40, f'${v:,.0f}', ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('algorithm_comparison.png', dpi=300)
plt.show()

## Step 9.4: Observations

1. **Tree Ensembles Outperform Linear Baselines**: **Extra Trees** and **Histogram Gradient Boosting** achieve the lowest RMSE (~$530–$550), substantially outperforming regularized linear regression (~$1,137) and Linear SVR (~$1,466).
2. **Capturing Non-Linear Gem Pricing**: Diamond price scales exponentially with carat weight and interacts heavily with cut and clarity. Non-linear tree architectures capture these interactions naturally without explicit polynomial feature generation.
3. **Speed vs. Memory Trade-Off**: Extra Trees takes longer and produces a large tree ensemble (>300 MB), whereas Histogram Gradient Boosting finishes in only ~33 seconds and produces a compact ~0.35 MB artifact with virtually identical accuracy (only 3% difference in RMSE), making it the ideal choice for production cloud deployment.

# Step 10: Train the Final Model
## Step 10.1: Select the Best Algorithm and Train on Full Data

In [ ]:
best_algorithm_name = summary.index[0]
best_algorithm = algorithms[best_algorithm_name]
print(f"Best Algorithm Selected: '{best_algorithm_name}' (Mean RMSE: ${summary.loc[best_algorithm_name, 'Mean RMSE ($)']:,.2f})")

final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', best_algorithm)
])

print('Training final model on 100% of the dataset...')
final_model.fit(X, y)
print('Final model training completed.')

## Step 10.3: Save the Trained Model

In [ ]:
MODEL_FILE = 'diamond_price_model_full_dataset.pkl'
with open(MODEL_FILE, 'wb') as f:
    pickle.dump(final_model, f)

print(f"Trained production pipeline saved to '{MODEL_FILE}'")

if IN_COLAB:
    from google.colab import files
    files.download(MODEL_FILE)

# Step 11: Execute the Application Phase
## Step 11.1: Take Input from User

In [ ]:
USE_INTERACTIVE_INPUT = False

example_diamond = {
    'carat': 1.01,
    'cut': 'Ideal',
    'color': 'G',
    'clarity': 'SI1',
    'depth': 61.8,
    'table': 56.0,
    'x': 6.44,
    'y': 6.41,
    'z': 3.97
}

def get_user_input():
    print('Enter Diamond Specifications:')
    d = {}
    d['carat'] = float(input('Carat Weight (0.2 - 5.0): '))
    d['cut'] = input('Cut (Fair, Good, Very Good, Premium, Ideal): ').strip()
    d['color'] = input('Color (D, E, F, G, H, I, J): ').strip()
    d['clarity'] = input('Clarity (IF, VVS1, VVS2, VS1, VS2, SI1, SI2, I1): ').strip()
    d['depth'] = float(input('Depth %: '))
    d['table'] = float(input('Table %: '))
    d['x'] = float(input('Length X (mm): '))
    d['y'] = float(input('Width Y (mm): '))
    d['z'] = float(input('Depth Z (mm): '))
    return d

input_diamond = get_user_input() if USE_INTERACTIVE_INPUT else example_diamond

print('Diamond Specifications for Inference:')
for k, v in input_diamond.items():
    print(f"  {k:<10}: {v}")

## Step 11.2: Convert User Input into Feature Vector
Format the dictionary into a DataFrame with the exact feature column ordering.

In [ ]:
feature_vector = pd.DataFrame([input_diamond])[numeric_features + categorical_features]
feature_vector

## Step 11.3: Load the Saved Model

In [ ]:
with open(MODEL_FILE, 'rb') as f:
    loaded_pipeline = pickle.load(f)

print(f"Model loaded from '{MODEL_FILE}': {loaded_pipeline.named_steps['regressor'].__class__.__name__}")

## Step 11.4: Model Prediction
Predict diamond market price with error bounds derived from cross-validation.

In [ ]:
predicted_price = loaded_pipeline.predict(feature_vector)[0]
best_rmse = summary.loc[best_algorithm_name, 'Mean RMSE ($)']

print(f"Predicted Diamond Market Price: ${predicted_price:,.2f}")
print(f"Expected Estimation Error (CV RMSE of {best_algorithm_name}): about ±${best_rmse:,.0f}")
print(f"Estimated Value Range: ${max(0, predicted_price - best_rmse):,.2f} to ${predicted_price + best_rmse:,.2f}")

# Step 12: Execute the Feedback Phase & Deployment Integration

After deploying the diamond valuation model to production, user interactions and appraiser feedback guide continuous system improvements:

1. **Operational Cloud Deployment**: The model has been packaged into an interactive web application using **Streamlit** and hosted live in the cloud. Appraisers and buyers can interactively adjust 4Cs parameters via sliders and select boxes.
2. **Hyperparameter Optimization**: Finer tuning of `learning_rate` and `max_leaf_nodes` for Histogram Gradient Boosting via `GridSearchCV` or Bayesian Optimization can yield further RMSE reductions.
3. **Segmented Valuation**: Segmenting the dataset by carat tiers (e.g. sub-1.0 ct vs. investment grade > 3.0 ct) allows dedicated expert models tailored to distinct market dynamics.
4. **Real-Time Market Feedback Loop**: Logging actual transaction prices alongside predicted prices enables automated retraining when diamond market exchange rates shift.

# Conclusion

In this assignment, we successfully evaluated **5 distinct machine learning algorithms** on a large real-world dataset (Full Dataset) using **10-Fold Cross-Validation**.

* **Top Performing Model**: `Tree Ensemble` achieved superior predictive performance with a low RMSE.
* **Clean Pipeline Architecture**: Enclosing `StandardScaler` and `OneHotEncoder` strictly within a `ColumnTransformer` inside the CV loop eliminated data leakage.
* **Cloud Web Application**: The trained model was deployed as an end-to-end web appraisal service.

> 🌐 **Live Cloud Deployment URL**: [https://gemval-diamond-ai.streamlit.app](https://gemval-diamond-ai.streamlit.app)